In [51]:
import datar.all as dr
from datar import f

import polars as pl

from pipda import register_verb, register_func

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
pl.Config.set_tbl_cell_alignment("CENTER")

data_dir = next(Path("/home/").glob("**/datar-polars/data"))

In [22]:
df_baseball = (
    dr.tibble(
        pl.read_csv(
            source=data_dir/"baseball.csv",
            columns=["Name", "Team", "Height", "Weight"],
        )
    )
    >> dr.mutate(Team = dr.as_factor(f.Team))
)

print(df_baseball >> dr.slice_head(4))

shape: (4, 4)
┌─────────────────┬──────┬────────┬────────┐
│       Name      ┆ Team ┆ Height ┆ Weight │
│       ---       ┆  --- ┆   ---  ┆   ---  │
│       str       ┆  cat ┆   i64  ┆   i64  │
╞═════════════════╪══════╪════════╪════════╡
│  Adam_Donachie  ┆  BAL ┆   74   ┆   180  │
│    Paul_Bako    ┆  BAL ┆   74   ┆   215  │
│ Ramon_Hernandez ┆  BAL ┆   72   ┆   210  │
│   Kevin_Millar  ┆  BAL ┆   72   ┆   210  │
└─────────────────┴──────┴────────┴────────┘


# <span style="color:#1E90FF">1. register_verb()</span>

## ``register_verb()`` can be used to change some conflict names
## For example: ``dr.filter_()`` conflict with Python built-in function filter()
## => Use registter_verb to bring it back as dr.filter()

In [23]:
dr.filter = register_verb(func=dr.filter_)

print(df_baseball >> dr.filter((f.Height > 75) & (f.Weight <= 200)) >> dr.slice_head(4))

shape: (4, 4)
┌────────────────┬──────┬────────┬────────┐
│      Name      ┆ Team ┆ Height ┆ Weight │
│       ---      ┆  --- ┆   ---  ┆   ---  │
│       str      ┆  cat ┆   i64  ┆   i64  │
╞════════════════╪══════╪════════╪════════╡
│   Kris_Benson  ┆  BAL ┆   76   ┆   195  │
│   James_Hoey   ┆  BAL ┆   78   ┆   200  │
│  Ryan_Sweeney  ┆  CWS ┆   76   ┆   200  │
│ Mike_MacDougal ┆  CWS ┆   76   ┆   195  │
└────────────────┴──────┴────────┴────────┘


# <span style="color:#1E90FF">2. register_func()</span>

## ``register_func()`` can be used to register a function from other packages into pipeable function

In [94]:
from scipy import stats
dr.shapiro = register_func(lambda x: [list(stats.shapiro(x))]) 
# Calculate shapiro then returns the output as [[W-statistic, p_value]]
# [[W-statistic, p_value]] is a 1-row List column, so that can be exploded into 2 rows

print(
    df_baseball
    >> dr.pipe(lambda f: f >> dr.reframe(
       # Use ``dr.pipe()`` with ``lambda f:`` to get eager direct access and bypass the polars expression, so that ``shapiro`` can work, 
            height_shapiro = dr.shapiro(f.Height), 
            weight_shapiro = dr.shapiro(f.Weight)
    ))
    >> dr.add_column(index=["W-statistic", "p_value"], _before=0)
)

shape: (2, 3)
┌─────────────┬────────────────┬────────────────┐
│    index    ┆ height_shapiro ┆ weight_shapiro │
│     ---     ┆       ---      ┆       ---      │
│     str     ┆       f64      ┆       f64      │
╞═════════════╪════════════════╪════════════════╡
│ W-statistic ┆    0.980529    ┆    0.988704    │
│   p_value   ┆   2.0754e-10   ┆    4.8153e-7   │
└─────────────┴────────────────┴────────────────┘


## Can use ``scipy.stats.shapiro`` without registering but need to reformulate the output as ``[list(...)]`` many times

In [95]:
print(
    df_baseball
    >> dr.pipe(lambda f: f >> dr.reframe(
       # Use ``dr.pipe()`` with ``lambda f:`` to get eager direct access and bypass the polars expression, so that ``shapiro`` can work, 
            height_shapiro = [list(stats.shapiro(f.Height))], 
            weight_shapiro = [list(stats.shapiro(f.Weight))]
    ))
    >> dr.add_column(index=["W-statistic", "p_value"], _before=0)
)

shape: (2, 3)
┌─────────────┬────────────────┬────────────────┐
│    index    ┆ height_shapiro ┆ weight_shapiro │
│     ---     ┆       ---      ┆       ---      │
│     str     ┆       f64      ┆       f64      │
╞═════════════╪════════════════╪════════════════╡
│ W-statistic ┆    0.980529    ┆    0.988704    │
│   p_value   ┆   2.0754e-10   ┆    4.8153e-7   │
└─────────────┴────────────────┴────────────────┘


## ``Numpy functions`` can be used to WITHOUT registering and reformulating outputs

In [96]:
import numpy as np

print(
    df_baseball
    >> dr.pipe(lambda f: f >> dr.reframe(
            height_quantiles=np.quantile(f.Height, q=[0.25, 0.5, 0.75]),
            weight_quantiles=np.quantile(f.Weight, q=[0.25, 0.5, 0.75]),
    ))
    >> dr.add_column(index=["Q1", "Q2", "Q3"], _before=0)
)

shape: (3, 3)
┌───────┬──────────────────┬──────────────────┐
│ index ┆ height_quantiles ┆ weight_quantiles │
│  ---  ┆        ---       ┆        ---       │
│  str  ┆        f64       ┆        f64       │
╞═══════╪══════════════════╪══════════════════╡
│   Q1  ┆       72.0       ┆       186.0      │
│   Q2  ┆       74.0       ┆       200.0      │
│   Q3  ┆       75.0       ┆       215.0      │
└───────┴──────────────────┴──────────────────┘


## Use np.apply_along_axis() for functions like scipy.stats.shapiro

In [97]:
print(
    df_baseball
    >> dr.pipe(lambda f: f >> dr.reframe(
        height_shapiro=np.apply_along_axis(stats.shapiro, 0, f.Height),
        weight_shapiro=np.apply_along_axis(stats.shapiro, 0, f.Height),
    ))
    >> dr.add_column(index=["W-statistic", "p_value"], _before=0)
)

shape: (2, 3)
┌─────────────┬────────────────┬────────────────┐
│    index    ┆ height_shapiro ┆ weight_shapiro │
│     ---     ┆       ---      ┆       ---      │
│     str     ┆       f64      ┆       f64      │
╞═════════════╪════════════════╪════════════════╡
│ W-statistic ┆    0.980529    ┆    0.980529    │
│   p_value   ┆   2.0754e-10   ┆   2.0754e-10   │
└─────────────┴────────────────┴────────────────┘
